## Imports


In [8]:
import numpy as np
import pandas as pd
import os

## Load board.npy from experiments

In [23]:
# Define path to board.npy
board_path = os.path.join('experiments', 'board.npy')

# Load the file 
board_data = np.load(board_path)
board_df = pd.DataFrame(board_data)
board_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,0,0,0,4,2,0,0,0,0,0,0,0,6,6,6,3
1,0,0,2,2,2,0,1,1,1,1,1,1,1,1,0,0
2,0,0,2,0,0,0,0,0,0,0,0,0,0,1,0,0
3,0,0,2,1,1,1,1,1,1,1,1,1,0,1,0,0
4,0,0,2,2,2,0,0,0,0,0,0,0,0,1,0,0
5,0,0,2,2,2,0,1,1,1,1,0,0,1,1,1,0
6,0,0,2,2,2,0,1,0,0,0,0,0,1,0,0,0
7,0,0,0,0,0,0,1,1,1,1,1,0,1,0,1,1
8,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0
9,0,0,0,0,0,1,1,1,1,1,1,0,1,1,1,0


In [67]:
import duckdb
from IPython.display import Markdown, display


# Query all experiments
results = duckdb.query("""
    SELECT 
        experiment.episode.nr as episode_number,
        experiment.episode.mode as mode,
        len(experiment.episode.steps) as num_steps,
        experiment.episode.steps[1].reward_to_go as total_reward
    FROM read_parquet('experiments/experiment_*.parquet')
    ORDER BY episode_number
""").df()

display(Markdown("# Reward in Episode"))
pd.set_option('display.max_rows', None)
results

# Reward in Episode

,episode_number,mode,num_steps,total_reward
0,100,validate,100,-100.0
1,200,validate,100,-100.0
2,300,validate,100,-100.0
3,400,validate,100,-100.0
4,500,validate,100,-100.0
5,600,validate,100,-100.0
6,700,validate,100,-100.0
7,800,validate,100,-100.0
8,900,validate,100,-100.0
9,1000,validate,100,-100.0


In [70]:
import duckdb

# Pfad zu deinen Dateien (nutzt Wildcards für alle Episoden)
parquet_files = "./experiments/experiment_*.parquet"

# DuckDB nutzen, um die Daten zu aggregieren
con = duckdb.connect()

query = f"""
SELECT * FROM(
    SELECT 
        "experiment"."episode"."nr" AS episode_number, 
        "experiment"."episode"."mode" AS episode_mode,  
        step.reward AS reward, 
        step.strategy AS strategy,
        COUNT(*) AS count
    FROM read_parquet('{parquet_files}')
    CROSS JOIN UNNEST("experiment"."episode"."steps") AS t(step) 
    GROUP BY episode_number, episode_mode, strategy, reward
    ORDER BY episode_number, episode_mode, strategy, reward)
    WHERE reward=-10
"""

result = con.execute(query).df()

result

# Optional: Als eine einzige CSV oder Parquet zusammenfassen
# con.execute(f"COPY ({query}) TO 'aggregated_rewards.csv' (HEADER, DELIMITER ',')")

,episode_number,episode_mode,reward,strategy,count
0,4700,validate,-10.0,greedy,64


In [57]:
import duckdb
pd.set_option('display.max_colwidth', None)
# Zeigt dir die Struktur der Datei an
duckdb.query(f"DESCRIBE SELECT * FROM read_parquet('experiments/experiment_000001.parquet')").df()

,column_name,column_type,null,key,default,extra
0,experiment,"STRUCT(world STRUCT(size_x INTEGER, size_y INTEGER, grid VARCHAR[][]), episode STRUCT(nr INTEGER, steps STRUCT(num INTEGER, pos_x SMALLINT, pos_y SMALLINT, ""action"" VARCHAR, reward_to_go FLOAT, strategy VARCHAR)[], ""mode"" VARCHAR), q_table FLOAT[][][])",YES,None,None,None


In [58]:
import pyarrow.parquet as pq

# Nimm einfach die erste Datei aus deinem Ordner
schema = pq.read_schema('experiments/experiment_000001.parquet')
print(schema.to_string())

experiment: struct<world: struct<size_x: int32, size_y: int32, grid: list<element: list<element: dictionary<valu (... 311 chars omitted)
  child 0, world: struct<size_x: int32, size_y: int32, grid: list<element: list<element: dictionary<values=string, ind (... 24 chars omitted)
      child 0, size_x: int32
      child 1, size_y: int32
      child 2, grid: list<element: list<element: dictionary<values=string, indices=int8, ordered=0>>>
          child 0, element: list<element: dictionary<values=string, indices=int8, ordered=0>>
              child 0, element: dictionary<values=string, indices=int8, ordered=0>
  child 1, episode: struct<nr: int32, steps: list<element: struct<num: int32, pos_x: int16, pos_y: int16, action: dictio (... 100 chars omitted)
      child 0, nr: int32
      child 1, steps: list<element: struct<num: int32, pos_x: int16, pos_y: int16, action: dictionary<values=string, indic (... 60 chars omitted)
          child 0, element: struct<num: int32, pos_x: int16, pos_y: 